In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", False)

# Methods

In [2]:
def calculate_metrics(true_labels, predicted_labels):
    """Calculate precision, recall, F1 score, and confusion matrix values.

    Args:
        true_labels (list of int): A list of true binary labels (0 or 1).
        predicted_labels (list of int): A list of predicted binary labels (0 or 1).

    Returns:
        dict: A dictionary containing the following metrics:
            - Precision (float): The proportion of true positive results in all positive results.
            - Recall (float): The proportion of true positive results in all pertinent cases.
            - F1 (float): The harmonic mean of precision and recall.
            - TP (int): The number of true positive cases.
            - FP (int): The number of false positive cases.
            - TN (int): The number of true negative cases.
            - FN (int): The number of false negative cases.
            - RATIO (float): The ratio of positive cases in the true labels.

    Examples:
        >>> calculate_metrics([1, 0, 1, 1], [1, 0, 1, 0])
        {'Precision': 1.0, 'Recall': 0.6666666666666666, 'F1': 0.8,
         'TP': 2, 'FP': 0, 'TN': 1, 'FN': 1, 'RATIO': 0.75}

        >>> calculate_metrics([0, 0, 0, 0], [0, 0, 0, 0])
        {'Precision': 0, 'Recall': 0, 'F1': 0, 'TP': 0, 'FP': 0,
         'TN': 4, 'FN': 0, 'RATIO': 0.0}
    """
    tp = sum(1 for t, p in zip(true_labels, predicted_labels, strict=False) if t == 1 and p == 1)
    fp = sum(1 for t, p in zip(true_labels, predicted_labels, strict=False) if t == 0 and p == 1)
    tn = sum(1 for t, p in zip(true_labels, predicted_labels, strict=False) if t == 0 and p == 0)
    fn = sum(1 for t, p in zip(true_labels, predicted_labels, strict=False) if t == 1 and p == 0)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    ratio = sum(true_labels) / len(true_labels) if len(true_labels) > 0 else 0

    return {
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn,
        "RATIO": ratio,
    }

def _process_and_display_all_results(df, languages, detection_type, sort_columns):
    # Loop through each language to process the results
    for language in languages:
        # Filter DataFrame based on language and detection_type
        filtered_df = df[(df["language"] == language) & (df["detection_type"] == detection_type)]

        # Ensure there is data for the given language and detection_type
        if filtered_df.empty:
            print(f"No {detection_type} data found for language: {language}")
            continue

        # Group data and calculate metrics
        group_columns = ["name", "language"] if detection_type == "REGEX" else ["name", "model_name", "language"]
        grouped_data = filtered_df.groupby(group_columns)
        results = []

        for keys, group_df in grouped_data:
            metrics = calculate_metrics(group_df["expected"], group_df["predicted"])
            keys_dict = dict(zip(group_columns, keys, strict=False))
            results.append({**keys_dict, **metrics})

        # Convert results to DataFrame
        results_df = pd.DataFrame(results)

        # Sort results according to provided columns and F1 score
        sort_columns = sort_columns if sort_columns else ["F1"] if detection_type == "REGEX" else ["name", "F1"]
        ascending = [c != "F1" for c in sort_columns]
        results_df.sort_values(by=sort_columns, ascending=ascending, inplace=True)

        # Save results to a CSV file
        filename = f"summary_test_results_{detection_type.lower()}_{language}.csv"
        results_df.to_csv(filename, index=False, sep=";")

        # Display results
        print(f"\n{language} {detection_type} Results:\n{results_df}")

def compare_sentences(df, language, detection_type, check="FP"):
    print("\n\n", "=" * 40, "\nChecking:", language, detection_type, check)

    # Filter DataFrame based on language and detection_type
    filtered_df = df[
        (df["language"] == language) & (df["detection_type"] == detection_type)
    ]

    # Ensure there is data for the given language and detection_type
    if filtered_df.empty:
        print(f"No {detection_type} data found for language: {language}")
        return

    total_false_positives = 0
    # Create an overview for each PII
    for name, group_df in filtered_df.groupby("name"):
        printed_name = False
        # Compare the sentences and print only False Positives
        for _, row in group_df.iterrows():
            original = row.get("sentence", "N/A")
            formatted = row.get("formatted_sentence", "N/A")
            expected = row.get("expected")
            predicted = row.get("predicted")
            test_type = row.get("test_type")
            entity = row.get("entity")
            reason = row.get("reason")

            if (
                (check == "FN" and not predicted and expected)
                or (check == "FP" and predicted and not expected)
                or (check == "TN" and not predicted and not expected)
                or (check == "TP" and predicted and expected)
            ):
                if not printed_name:
                    print(f"\nOverview for PII: {name}")
                    printed_name = True

                print(f"Type: {test_type}, Entity: {entity}")
                print(f"Expected: {expected}, Original:  {original}")
                print(f"Predicted: {predicted}, Formatted: {formatted}")
                print(f"Reason: {reason}")
                print("-" * 40)
                total_false_positives += 1
    print(f"Total {check}: {total_false_positives}")

In [3]:
df = pd.read_csv("all_test_results.csv", sep=";")

# Every PII Based Masking (PrivateAI)

In [4]:
df_compare = df[df["name"] != "PersonAndOrganisation"]
df_compare = df[df["name"] != "Organisation"]
df_compare = df_compare[df_compare["detection_type"] == "PrivateAI"]

df_dutch = df_compare[df_compare["language"] == "NL"]
df_english = df_compare[df_compare["language"] == "EN"]

print("Dutch:", calculate_metrics(df_dutch["expected"], df_dutch["predicted"]))
print("English:", calculate_metrics(df_english["expected"], df_english["predicted"]))

Dutch: {'Precision': 1.0, 'Recall': 0.8809523809523809, 'F1': 0.9367088607594937, 'TP': 148, 'FP': 0, 'TN': 120, 'FN': 20, 'RATIO': 0.5833333333333334}
English: {'Precision': 1.0, 'Recall': 0.8783068783068783, 'F1': 0.9352112676056338, 'TP': 166, 'FP': 0, 'TN': 135, 'FN': 23, 'RATIO': 0.5833333333333334}


In [15]:
_process_and_display_all_results(df, ["EN", "NL"], detection_type="PrivateAI", sort_columns=["F1"])


EN PrivateAI Results:
                                        name model_name language  Precision    Recall        F1  TP  FP  TN  FN     RATIO
0                                    Address     'Best'       EN        1.0  1.000000  1.000000   7   0   5   0  0.583333
1                AustralianBankAccountNumber     'Best'       EN        1.0  1.000000  1.000000   7   0   5   0  0.583333
2             AustralianDriversLicenseNumber     'Best'       EN        1.0  1.000000  1.000000   7   0   5   0  0.583333
3                   AustralianPassportNumber     'Best'       EN        1.0  1.000000  1.000000   7   0   5   0  0.583333
4                    AustralianTaxFileNumber     'Best'       EN        1.0  1.000000  1.000000   7   0   5   0  0.583333
5                                   Birthday     'Best'       EN        1.0  1.000000  1.000000   7   0   5   0  0.583333
6                  CanadianBankAccountNumber     'Best'       EN        1.0  1.000000  1.000000   7   0   5   0  0.583333
7

## FN and FP

In [14]:

compare_sentences(df, "NL", detection_type="PrivateAI", check='FN')
compare_sentences(df, "EN", detection_type="PrivateAI", check='FN')
compare_sentences(df, "NL", detection_type="PrivateAI", check='FP')
compare_sentences(df, "EN", detection_type="PrivateAI", check='FP')



Checking: NL PrivateAI FN

Overview for PII: Address
Type: Edge, Entity: Smalle Pad
Expected: 1, Original:  De nieuwe winkel op het Smalle Pad is vandaag geopend.
Predicted: 0, Formatted: De nieuwe winkel op het Hostal São is vandaag geopend. 
Reason: Expected the mask 'LOCATION_ADDRESS_STREET' to be presented in the formatted sentence, but it was not found in: '['LOCATION']'.
----------------------------------------

Overview for PII: Birthday
Type: True, Entity: 15 maart
Expected: 1, Original:  Mijn zusje is op 15 maart jarig en we organiseren een groot feest.
Predicted: 0, Formatted: Mijn zusje is op 15 maart jarig en we organiseren een groot feest. 
Reason: Expected the entity '15 maart' to be removed in the formatted sentence, but it was found in: 'Mijn zusje is op 15 maart jarig en we organiseren een groot feest. '.
----------------------------------------

Overview for PII: DutchIdentityDocumentNumber
Type: True, Entity: MN0PQR2S3
Expected: 1, Original:  Ik heb mijn identiteit

# Every PII Based Masking (PII_Everything)

In [ ]:
df_compare = df[df["name"] != "PersonAndOrganisation"]
df_compare = df_compare[df_compare["detection_type"] == "Everything"]

df_dutch = df_compare[df_compare["language"] == "NL"]
df_english = df_compare[df_compare["language"] == "EN"]

print("Dutch:", calculate_metrics(df_dutch["expected"], df_dutch["predicted"]))
print("English:", calculate_metrics(df_english["expected"], df_english["predicted"]))

Dutch: {'Precision': 1.0, 'Recall': 0.8809523809523809, 'F1': 0.9367088607594937, 'TP': 148, 'FP': 0, 'TN': 120, 'FN': 20, 'RATIO': 0.5833333333333334}
English: {'Precision': 1.0, 'Recall': 0.8783068783068783, 'F1': 0.9352112676056338, 'TP': 166, 'FP': 0, 'TN': 135, 'FN': 23, 'RATIO': 0.5833333333333334}


In [ ]:
_process_and_display_all_results(df, ["EN", "NL"], detection_type="Everything", sort_columns=["F1"])


EN PrivateAI Results:
                                        name model_name language  Precision    Recall        F1  TP  FP  TN  FN     RATIO
0                                    Address     'Best'       EN        1.0  1.000000  1.000000   7   0   5   0  0.583333
1                AustralianBankAccountNumber     'Best'       EN        1.0  1.000000  1.000000   7   0   5   0  0.583333
2             AustralianDriversLicenseNumber     'Best'       EN        1.0  1.000000  1.000000   7   0   5   0  0.583333
3                   AustralianPassportNumber     'Best'       EN        1.0  1.000000  1.000000   7   0   5   0  0.583333
4                    AustralianTaxFileNumber     'Best'       EN        1.0  1.000000  1.000000   7   0   5   0  0.583333
5                                   Birthday     'Best'       EN        1.0  1.000000  1.000000   7   0   5   0  0.583333
6                  CanadianBankAccountNumber     'Best'       EN        1.0  1.000000  1.000000   7   0   5   0  0.583333
7

## FN and FP

In [ ]:

compare_sentences(df, "NL", detection_type="Everything", check='FN')
compare_sentences(df, "EN", detection_type="Everything", check='FN')
compare_sentences(df, "NL", detection_type="Everything", check='FP')
compare_sentences(df, "EN", detection_type="Everything", check='FP')



Checking: NL PrivateAI FN

Overview for PII: Address
Type: Edge, Entity: Smalle Pad
Expected: 1, Original:  De nieuwe winkel op het Smalle Pad is vandaag geopend.
Predicted: 0, Formatted: De nieuwe winkel op het Hostal São is vandaag geopend. 
Reason: Expected the mask 'LOCATION_ADDRESS_STREET' to be presented in the formatted sentence, but it was not found in: '['LOCATION']'.
----------------------------------------

Overview for PII: Birthday
Type: True, Entity: 15 maart
Expected: 1, Original:  Mijn zusje is op 15 maart jarig en we organiseren een groot feest.
Predicted: 0, Formatted: Mijn zusje is op 15 maart jarig en we organiseren een groot feest. 
Reason: Expected the entity '15 maart' to be removed in the formatted sentence, but it was found in: 'Mijn zusje is op 15 maart jarig en we organiseren een groot feest. '.
----------------------------------------

Overview for PII: DutchIdentityDocumentNumber
Type: True, Entity: MN0PQR2S3
Expected: 1, Original:  Ik heb mijn identiteit

# REGEX Based PII Masking

In [7]:
_process_and_display_all_results(df, ["EN", "NL"], detection_type="REGEX", sort_columns=["F1"])

No REGEX data found for language: EN
No REGEX data found for language: NL


## FN and FP

In [8]:
compare_sentences(df, "NL", detection_type="REGEX", check="FN")
compare_sentences(df, "EN", detection_type="REGEX", check="FN")
compare_sentences(df, "NL", detection_type="REGEX", check="FP")
compare_sentences(df, "EN", detection_type="REGEX", check="FP")



Checking: NL REGEX FN
No REGEX data found for language: NL


Checking: EN REGEX FN
No REGEX data found for language: EN


Checking: NL REGEX FP
No REGEX data found for language: NL


Checking: EN REGEX FP
No REGEX data found for language: EN


# NER Based PII Masking

In [9]:
_process_and_display_all_results(df, ["EN", "NL"], detection_type="NER", sort_columns=["name", "F1"])

No NER data found for language: EN
No NER data found for language: NL


## FN and FP

In [10]:
compare_sentences(df, "NL", detection_type="NER", check='FN')
compare_sentences(df, "EN", detection_type="NER", check='FN')
compare_sentences(df, "NL", detection_type="NER", check='FP')
compare_sentences(df, "EN", detection_type="NER", check='FP')



Checking: NL NER FN
No NER data found for language: NL


Checking: EN NER FN
No NER data found for language: EN


Checking: NL NER FP
No NER data found for language: NL


Checking: EN NER FP
No NER data found for language: EN


## TN and TP (Correctly classified sentences)

## REGEX

In [11]:
compare_sentences(df, "NL", detection_type="REGEX", check="TN")
compare_sentences(df, "EN", detection_type="REGEX", check="TN")
compare_sentences(df, "NL", detection_type="REGEX", check="TP")
compare_sentences(df, "EN", detection_type="REGEX", check="TP")



Checking: NL REGEX TN
No REGEX data found for language: NL


Checking: EN REGEX TN
No REGEX data found for language: EN


Checking: NL REGEX TP
No REGEX data found for language: NL


Checking: EN REGEX TP
No REGEX data found for language: EN


## NER

In [12]:
compare_sentences(df, "NL", detection_type="NER", check="TN")
compare_sentences(df, "EN", detection_type="NER", check="TN")
compare_sentences(df, "NL", detection_type="NER", check="TP")
compare_sentences(df, "EN", detection_type="NER", check="TP")



Checking: NL NER TN
No NER data found for language: NL


Checking: EN NER TN
No NER data found for language: EN


Checking: NL NER TP
No NER data found for language: NL


Checking: EN NER TP
No NER data found for language: EN


## PII Everything

In [13]:
compare_sentences(df, "NL", detection_type="Everything", check="TN")
compare_sentences(df, "EN", detection_type="Everything", check="TN")
compare_sentences(df, "NL", detection_type="Everything", check="TP")
compare_sentences(df, "EN", detection_type="Everything", check="TP")



Checking: NL Everything TN
No Everything data found for language: NL


Checking: EN Everything TN
No Everything data found for language: EN


Checking: NL Everything TP
No Everything data found for language: NL


Checking: EN Everything TP
No Everything data found for language: EN
